# FAVAR Synthetic Walkthrough

This notebook is a complete, reproducible walkthrough of the `favar` package using synthetic macroeconomic data.

You will:

1. verify that `favar` is installed and importable;
2. generate a synthetic macroeconomic panel;
3. transform raw time series into model-ready inputs;
4. define the information panel `X`, observed variables `Y`, and slow-moving variables;
5. select the lag order of the augmented FAVAR system;
6. estimate the FAVAR model;
7. inspect the summary output;
8. produce forecasts with confidence intervals;
9. compute impulse response functions;
10. inspect residual autocorrelation diagnostics.

The example is intentionally self-contained: no external data files are required.

## 1. Installation Check

If the package is available on PyPI, install it with:

```bash
pip install favar
```

When working from a local repository checkout, install it with:

```bash
pip install -e ..
```

The next cell verifies that the current Python kernel can import the package.

In [1]:
import importlib
import sys

try:
    favar_pkg = importlib.import_module("favar")
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "The 'favar' package is not installed in this kernel. "
        "Install it with 'pip install favar' or, from the repository, 'pip install -e ..'."
    ) from exc

from favar import FAVAR

print("Python:", sys.version.split()[0])
print("favar:", getattr(favar_pkg, "__version__", "unknown"))
print("import check: ok")

Python: 3.12.4
favar: 0.1.0
import check: ok


## 2. Generate Synthetic Macroeconomic Data

The synthetic data-generating process has two latent macroeconomic factors and one policy-rate variable. The raw panel contains real activity levels, price indexes, financial spreads, and the policy rate. This lets us demonstrate realistic preprocessing choices before estimation.

In [2]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(2026)
T = 180
burn = 120
K_TRUE = 2
state_dim = K_TRUE + 1

A = np.array([
    [0.55, 0.10, 0.06],
    [0.08, 0.50, 0.04],
    [0.25, 0.15, 0.65],
])
spectral_radius = np.max(np.abs(np.linalg.eigvals(A)))
if spectral_radius >= 0.95:
    A *= 0.95 / spectral_radius

B = np.array([
    [0.70, 0.00, 0.00],
    [0.20, 0.60, 0.00],
    [0.10, 0.25, 0.55],
])

states = np.zeros((T + burn, state_dim))
for t in range(1, T + burn):
    states[t] = A @ states[t - 1] + B @ rng.normal(size=state_dim)
states = states[burn:]

activity_factor = states[:, 0]
price_factor = states[:, 1]
policy_rate = 5 + states[:, 2] + rng.normal(scale=0.10, size=T)

dates = pd.period_range("2005-01", periods=T, freq="M").to_timestamp()
raw = pd.DataFrame(index=dates)

# Slow-moving level series.
for i in range(8):
    growth = 0.003 + 0.010 * activity_factor + rng.normal(scale=0.006, size=T)
    raw[f"real_activity_{i+1}"] = 100 * np.exp(np.cumsum(growth))

for i in range(6):
    inflation_component = 0.002 + 0.006 * price_factor + rng.normal(scale=0.004, size=T)
    raw[f"price_index_{i+1}"] = 100 * np.exp(np.cumsum(inflation_component))

# Fast-moving financial series.
for i in range(8):
    raw[f"financial_spread_{i+1}"] = (
        1.5 + 0.35 * policy_rate + 0.20 * activity_factor + rng.normal(scale=0.25, size=T)
    )

raw["policy_rate"] = policy_rate
raw.head()

,real_activity_1,real_activity_2,real_activity_3,real_activity_4,real_activity_5,real_activity_6,real_activity_7,real_activity_8,price_index_1,price_index_2,...,price_index_6,financial_spread_1,financial_spread_2,financial_spread_3,financial_spread_4,financial_spread_5,financial_spread_6,financial_spread_7,financial_spread_8,policy_rate
2005-01-01,100.081155,100.296710,100.633439,100.864236,100.961995,101.267361,100.011744,100.211223,100.590279,100.665557,...,100.713048,3.445112,3.491556,3.431495,3.755008,3.329578,3.348123,3.310122,3.408302,4.666628
2005-02-01,102.487881,102.697025,102.773309,103.635571,103.107546,103.873785,102.471778,101.716388,101.592544,100.707699,...,101.602785,3.830199,3.338772,3.298392,3.268479,3.612524,3.525423,3.421869,3.703251,4.873267
2005-03-01,103.992276,104.432496,103.938812,105.732641,104.670097,105.295192,104.464564,103.961827,102.215827,101.296435,...,102.652082,3.548382,3.371461,3.805600,3.542228,3.955767,3.261804,3.369878,3.965745,5.393808
2005-04-01,104.501570,104.663070,104.596024,107.452325,106.255455,106.288968,106.073324,105.769729,102.664594,101.383084,...,103.296607,3.227617,3.210450,3.262494,3.416289,3.217144,3.311250,3.675304,3.290480,5.113252
2005-05-01,104.605896,103.793706,105.144209,107.241427,106.262927,106.620753,105.822133,106.386690,102.920291,101.242606,...,103.961624,2.571740,3.296206,3.100461,3.235805,2.976722,3.311178,2.735441,2.714205,4.490112


## 3. Transform the Raw Series

A FAVAR model should receive economically meaningful transformations. In this example:

- real activity levels are converted to monthly log growth rates;
- price indexes are converted to inflation rates;
- financial spreads are used in levels;
- the policy rate enters `Y` in levels.

The package can standardize the information panel internally, but it does not choose these economic transformations for you.

In [3]:
X = pd.DataFrame(index=raw.index)
slow_columns = []

for col in [c for c in raw.columns if c.startswith("real_activity_")]:
    name = "dlog_" + col
    X[name] = 100 * np.log(raw[col]).diff()
    slow_columns.append(name)

for col in [c for c in raw.columns if c.startswith("price_index_")]:
    name = "inflation_" + col.replace("price_index_", "")
    X[name] = 100 * np.log(raw[col]).diff()
    slow_columns.append(name)

for col in [c for c in raw.columns if c.startswith("financial_spread_")]:
    X[col] = raw[col]

Y = pd.DataFrame(index=raw.index)
Y["activity_growth"] = X["dlog_real_activity_1"]
Y["inflation"] = X["inflation_1"]
Y["policy_rate"] = raw["policy_rate"]

combined = pd.concat([X, Y], axis=1).dropna()
X = combined[X.columns]
Y = combined[Y.columns]

print("X shape:", X.shape)
print("Y shape:", Y.shape)
print("Number of slow-moving variables:", len(slow_columns))
X.head()

X shape: (179, 22)
Y shape: (179, 3)
Number of slow-moving variables: 14


,dlog_real_activity_1,dlog_real_activity_2,dlog_real_activity_3,dlog_real_activity_4,dlog_real_activity_5,dlog_real_activity_6,dlog_real_activity_7,dlog_real_activity_8,inflation_1,inflation_2,...,inflation_5,inflation_6,financial_spread_1,financial_spread_2,financial_spread_3,financial_spread_4,financial_spread_5,financial_spread_6,financial_spread_7,financial_spread_8
2005-02-01,2.376315,2.365026,2.104108,2.710521,2.102842,2.541240,2.429980,1.490824,0.991453,0.041855,...,0.673972,0.879559,3.830199,3.338772,3.298392,3.268479,3.612524,3.525423,3.421869,3.703251
2005-03-01,1.457207,1.675774,1.127670,2.003303,1.504089,1.359120,1.926049,2.183535,0.611638,0.582896,...,1.294409,1.027448,3.548382,3.371461,3.805600,3.542228,3.955767,3.261804,3.369878,3.965745
2005-04-01,0.488547,0.220545,0.630317,1.613361,1.503268,0.939374,1.528268,1.724058,0.438078,0.085503,...,0.615870,0.625910,3.227617,3.210450,3.262494,3.416289,3.217144,3.311250,3.675304,3.290480
2005-05-01,0.099782,-0.834100,0.522729,-0.196464,0.007032,0.311668,-0.237090,0.581612,0.248750,-0.138658,...,-0.531997,0.641730,2.571740,3.296206,3.100461,3.235805,2.976722,3.311178,2.735441,2.714205
2005-06-01,0.481744,0.465110,-0.330230,0.128607,0.531919,-0.957947,-0.394782,-0.104495,0.456054,0.953267,...,0.994174,0.587856,3.711703,3.176810,3.233982,3.124137,3.297788,3.191511,3.587477,3.450249


## 4. Select the Lag Order

Before estimating a fixed lag order, inspect information criteria for the augmented FAVAR system. The table marks each criterion minimum with an asterisk.

In [4]:
order_model = FAVAR(
    X=X,
    Y=Y,
    policy_var="policy_rate",
    k_factors=2,
    slow_columns=slow_columns,
    standardize=True,
)

order_selection = order_model.select_order(maxlags=6)
print(order_selection.summary())

FAVAR Lag Order Selection (* highlights the minimums)
    AIC     BIC      FPE       HQIC 
------------------------------------
0  -7.023  -6.932  0.0008911  -6.986
1 -8.191* -7.645* 0.0002771* -7.970*
2  -8.040  -7.037  0.0003227  -7.633
3  -7.964  -6.506  0.0003486  -7.373
4  -7.786  -5.872  0.0004181  -7.009
5  -7.689  -5.319  0.0004632  -6.727
6  -7.647  -4.821  0.0004871  -6.500
------------------------------------


## 5. Estimate the FAVAR Model

We now estimate a two-factor FAVAR with two lags. The policy rate is ordered last for recursive policy-shock identification.

In [5]:
model = FAVAR(
    X=X,
    Y=Y,
    policy_var="policy_rate",
    k_factors=2,
    slow_columns=slow_columns,
    standardize=True,
)

results = model.fit(lags=2)
print(results.summary())

  Summary of FAVAR Regression Results   
Model:                           FAVAR
Estimator:                Two-step PCA
VAR method:                        OLS
Date:               Sun, 28, Jun, 2026
Time:                         21:26:29
--------------------------------------------------------------------
No. of Equations:               5    BIC:                  -7.10035
Nobs:                         177    HQIC:                 -7.68702
Log likelihood:        -485.03561    FPE:                   0.00031
AIC:                     -8.08729    Det(Omega_mle):        0.00023
--------------------------------------------------------------------
FAVAR Model Information
No. of factors:                                                    2
No. of X variables:                                               22
No. of observed Y variables:                                       3
No. of slow-moving variables:                                     14
Policy variable:                                      

The fitted result stores the estimated factors, the system ordering, variance shares from the principal components step, and stability diagnostics.

In [6]:
print("System order:", results.order)
print("PC variance shares:", np.round(results.explained_variance_ratio, 3))
print("Stable system:", results.is_stable())

factors = pd.DataFrame(results.factors, index=X.index, columns=results.factor_names)
factors.head()

System order: ['F1', 'F2', 'activity_growth', 'inflation', 'policy_rate']
PC variance shares: [0.554 0.117]
Stable system: True


,F1,F2
2005-02-01,0.361823,-0.392514
2005-03-01,0.617388,-0.133294
2005-04-01,0.979132,-0.056291
2005-05-01,1.583039,-0.134025
2005-06-01,1.432212,0.380137


## 6. Forecast with Confidence Intervals

`forecast()` returns forecasts for the observed variables in `Y`. The next cell produces a 12-month forecast with 95% confidence intervals.

In [7]:
forecast = results.forecast(steps=12, confidence_level=0.95)
forecast.head()

,activity_growth,activity_growth_lower,activity_growth_upper,inflation,inflation_lower,inflation_upper,policy_rate,policy_rate_lower,policy_rate_upper
2020-01-01,1.092360,-0.696589,2.881309,0.268886,-0.771633,1.309405,5.958249,4.695544,7.220953
2020-02-01,0.895106,-1.061838,2.852050,0.273732,-0.827571,1.375036,5.859277,4.240543,7.478010
2020-03-01,0.708559,-1.331140,2.748258,0.307935,-0.811293,1.427164,5.797213,3.985957,7.608469
2020-04-01,0.606334,-1.474791,2.687459,0.306723,-0.818968,1.432413,5.704354,3.773623,7.635084
2020-05-01,0.525462,-1.580983,2.631907,0.303646,-0.826292,1.433583,5.617501,3.607319,7.627683


In [8]:
# Compact view for the policy rate.
forecast[["policy_rate", "policy_rate_lower", "policy_rate_upper"]].round(3)

,policy_rate,policy_rate_lower,policy_rate_upper
2020-01-01,5.958,4.696,7.221
2020-02-01,5.859,4.241,7.478
2020-03-01,5.797,3.986,7.608
2020-04-01,5.704,3.774,7.635
2020-05-01,5.618,3.607,7.628
2020-06-01,5.537,3.474,7.599
2020-07-01,5.467,3.369,7.565
2020-08-01,5.408,3.286,7.529
2020-09-01,5.358,3.221,7.495
2020-10-01,5.317,3.169,7.464


## 7. System Impulse Response Functions

`impulse_response()` computes responses of the augmented FAVAR system. The example below scales the policy shock so that the impact response of the policy rate is 0.25.

In [9]:
irf_y = results.impulse_response(
    periods=36,
    shock="policy_rate",
    impulse_size=0.25,
    include_factors=False,
)
irf_y.head()

,activity_growth,inflation,policy_rate
period,,,
0,0.000000,0.000000,0.250000
1,0.026734,-0.001676,0.148769
2,0.042090,0.010141,0.110901
3,0.043098,0.011549,0.088025
4,0.036896,0.010345,0.071593


## 8. Panel-Projected Impulse Response Functions

`panel_impulse_response()` maps system responses back to selected series in the large information panel `X` using the estimated measurement equation.

In [10]:
irf_x = results.panel_impulse_response(
    periods=36,
    shock="policy_rate",
    impulse_size=0.25,
    columns=["dlog_real_activity_1", "inflation_1", "financial_spread_1"],
    scale="original",
)
irf_x.head()

,dlog_real_activity_1,inflation_1,financial_spread_1
period,,,
0,3.533451e-17,3.903277e-18,0.088472
1,2.673410e-02,-1.675691e-03,0.059663
2,4.209005e-02,1.014067e-02,0.049518
3,4.309829e-02,1.154888e-02,0.041158
4,3.689611e-02,1.034522e-02,0.033576


## 9. Forecast and IRF Plots

The following plots provide a quick visual check of the policy-rate forecast and the impulse responses of the observed variables.

In [11]:
import matplotlib.pyplot as plt
from IPython.display import display

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

policy_fc = forecast[["policy_rate", "policy_rate_lower", "policy_rate_upper"]]
axes[0].plot(Y.index[-36:], Y["policy_rate"].iloc[-36:], label="observed", color="black")
axes[0].plot(policy_fc.index, policy_fc["policy_rate"], label="forecast", color="tab:blue")
axes[0].fill_between(
    policy_fc.index,
    policy_fc["policy_rate_lower"],
    policy_fc["policy_rate_upper"],
    color="tab:blue",
    alpha=0.15,
)
axes[0].set_title("Policy-rate forecast")
axes[0].legend()

axes[1].axhline(0, color="black", linewidth=0.8)
for col in irf_y.columns:
    axes[1].plot(irf_y.index, irf_y[col], label=col)
axes[1].set_title("Observed-variable IRFs")
axes[1].set_xlabel("months")
axes[1].legend()

plt.tight_layout()
display(fig)
plt.close(fig)

<Figure size 1200x400 with 2 Axes>

## 10. Residual Autocorrelation Diagnostics

`plot_acorr()` shows residual autocorrelations and cross-correlations for the augmented FAVAR system. Dashed lines are $2 / \sqrt{T}$ bounds.

In [12]:
from IPython.display import display

fig = results.plot_acorr(nlags=10)
fig.set_size_inches(10, 10)
display(fig)
plt.close(fig)

<Figure size 1000x1000 with 25 Axes>

## 11. Exercise

Change `k_factors` from 2 to 3 or change `lags=2` to `lags=4`. Compare model stability, impulse responses, and forecast intervals.

In [13]:
model_alt = FAVAR(
    X=X,
    Y=Y,
    policy_var="policy_rate",
    k_factors=3,
    slow_columns=slow_columns,
    standardize=True,
)

results_alt = model_alt.fit(lags=2)
print("Alternative model is stable:", results_alt.is_stable())
print("Alternative system order:", results_alt.order)
results_alt.forecast(steps=4, confidence_level=0.95).round(3)

Alternative model is stable: True
Alternative system order: ['F1', 'F2', 'F3', 'activity_growth', 'inflation', 'policy_rate']


,activity_growth,activity_growth_lower,activity_growth_upper,inflation,inflation_lower,inflation_upper,policy_rate,policy_rate_lower,policy_rate_upper
2020-01-01,0.948,-0.846,2.742,0.281,-0.766,1.327,5.859,4.592,7.125
2020-02-01,0.807,-1.161,2.774,0.250,-0.858,1.358,5.671,4.050,7.291
2020-03-01,0.594,-1.457,2.644,0.286,-0.840,1.411,5.624,3.804,7.445
2020-04-01,0.536,-1.559,2.631,0.281,-0.852,1.413,5.549,3.604,7.494


## Common Pitfall

A common source of errors is misalignment between `X` and `Y`, especially after log differences or missing-value removal. Always check the final inputs before estimation.

In [14]:
assert X.index.equals(Y.index)
assert not X.isna().any().any()
assert not Y.isna().any().any()
assert "policy_rate" in Y.columns
assert all(col in X.columns for col in slow_columns)

print("Inputs are ready for estimation.")

Inputs are ready for estimation.
